In [69]:
import pandas as pd
import numpy as np

train = pd.read_csv('train_data.csv')
test = pd.read_csv('test_data.csv')

ls = ['Race','Marital Status','T Stage','N Stage','6th Stage','differentiate','Grade','A Stage','Estrogen Status','Progesterone Status','T_N_Stage','Hormone_Status']

train['6th Stage'] = train['6th Stage'].fillna(train['6th Stage'].value_counts()[0])
for i in ls:
    train[i] = train[i].astype('category')
    test[i] = train[i].astype('category')

/tmp/ipykernel_21787/1411287164.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  train['6th Stage'] = train['6th Stage'].fillna(train['6th Stage'].value_counts()[0])


In [70]:
from catboost import  CatBoostClassifier

model = CatBoostClassifier(iterations = 600, rsm = 0.85, learning_rate = 0.26, l2_leaf_reg = 0.3, max_depth = 8,cat_features = ['Race','Marital Status','T Stage','N Stage','6th Stage','differentiate','Grade','A Stage','Estrogen Status','Progesterone Status','T_N_Stage','Hormone_Status'])
model.fit(train.drop(['Status','ID'], axis = 1),train['Status'])

0:	learn: 0.5234503	total: 59.2ms	remaining: 35.5s
1:	learn: 0.4388860	total: 87ms	remaining: 26s
2:	learn: 0.4031703	total: 108ms	remaining: 21.4s
3:	learn: 0.3667543	total: 119ms	remaining: 17.8s
4:	learn: 0.3569790	total: 124ms	remaining: 14.8s
5:	learn: 0.3539781	total: 126ms	remaining: 12.5s
6:	learn: 0.3390241	total: 139ms	remaining: 11.8s
7:	learn: 0.3253893	total: 151ms	remaining: 11.1s
8:	learn: 0.3179828	total: 162ms	remaining: 10.7s
9:	learn: 0.3024536	total: 173ms	remaining: 10.2s
10:	learn: 0.2866412	total: 183ms	remaining: 9.8s
11:	learn: 0.2785882	total: 195ms	remaining: 9.55s
12:	learn: 0.2667849	total: 207ms	remaining: 9.33s
13:	learn: 0.2635111	total: 219ms	remaining: 9.18s
14:	learn: 0.2576953	total: 232ms	remaining: 9.04s
15:	learn: 0.2454650	total: 245ms	remaining: 8.94s
16:	learn: 0.2428757	total: 257ms	remaining: 8.81s
17:	learn: 0.2381388	total: 268ms	remaining: 8.67s
18:	learn: 0.2378592	total: 272ms	remaining: 8.32s
19:	learn: 0.2304384	total: 284ms	remaining:

CatBoostClassifier(cat_features=['Race', 'Marital Status', 'T Stage', 'N Stage', '6th Stage', 'differentiate', 'Grade', 'A Stage', 'Estrogen Status', 'Progesterone Status', 'T_N_Stage', 'Hormone_Status'], iterations=600, l2_leaf_reg=0.3, learning_rate=0.26, max_depth=8, rsm=0.85)

In [71]:
subtask5_answers = model.predict(test.drop('ID',axis=1))

In [84]:
subtask1_answer = ['Normal' if test['GFR'].iloc[i] >= 90 else 'Mildly Decreased' for i in range(0,len(test))]

train_stage = train['T Stage'].astype(str).str.strip().str.upper()
test_stage = test['T Stage'].astype(str).str.strip().str.upper()
counts = train_stage.value_counts().to_dict()
subtask4_answer = [int(counts.get(x, 0)) for x in test_stage]

In [73]:
subtask2_answer = []
for i in test['Serum Creatinine']:
    if i <= train['Serum Creatinine'].describe().loc['25%']:
        subtask2_answer.append('Very Low')
    elif train['Serum Creatinine'].describe().loc['25%'] < i <= train['Serum Creatinine'].describe().loc['50%']:
        subtask2_answer.append('Low')
    elif train['Serum Creatinine'].describe().loc['50%'] < i <= train['Serum Creatinine'].describe().loc['75%']:
        subtask2_answer.append('High')
    else:
        subtask2_answer.append('Very High')

In [77]:
subtask3 = [1 if x > train['BMI'].median() else 0 for x in test['BMI']]

In [79]:
final_result = []
ids = []
subtasks = [1,2,3,4,5] * len(test)
for i in range(0, len(test)):
    ids.extend([test['ID'].iloc[i]]*5)
    final_result.append(subtask1_answer[i])
    final_result.append(subtask2_answer[i])
    final_result.append(subtask3[i])
    final_result.append(subtask4_answer[i])
    final_result.append(subtask5_answers[i])


In [80]:
pd.DataFrame({
    'subtaskID':subtasks,
    'datapointID':ids,
    'answer':final_result
}).to_csv('submission.csv',index=False)